In [66]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [67]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(
    model='google_genai:gemini-3.5-flash',
    temperature=0.0
)

In [68]:
from utils import show_prompt

from prompts import (
    LS_DESCRIPTION,
    WRITE_FILE_DESCRIPTION,
    READ_FILE_DESCRIPTION
)

show_prompt(LS_DESCRIPTION)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│  List all files in the virtual filesystem stored in agent state.                                                │
│                                                                                                                 │
│  Shows what files currently exist in agent memory. Use this to orient yourself before other file operations     │
│  and maintain awareness of your file organization.                                                              │
│                                                                                                                 │
│  No parameters required - simply call ls() to see all available files.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [69]:
show_prompt(READ_FILE_DESCRIPTION)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│  Read content from a file in the virtual filesystem with optional pagination.                                   │
│                                                                                                                 │
│  This tool returns file content with line numbers (like `cat -n`) and supports reading large files in chunks    │
│  to avoid context overflow.                                                                                     │
│                                                                                                                 │
│  Parameters:                                                                                                    │
│  - file_path (required): Path to the file you want to read                                                      │
│  - offset (optional, default=0): Line number to start reading from                                              │
│  - limit (optional, default=2000): Maximum number of lines to read                                              │
│                                                                                                                 │
│  Essential before making any edits to understand existing content. Always read a file before editing it.        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [70]:
show_prompt(WRITE_FILE_DESCRIPTION)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│  Create a new file or completely overwrite an existing file in the virtual filesystem.                          │
│                                                                                                                 │
│  This tool creates new files or replaces entire file contents. Use for initial file creation or complete        │
│  rewrites. Files are stored persistently in agent state.                                                        │
│                                                                                                                 │
│  Parameters:                                                                                                    │
│  - file_path (required): Path where the file should be created/overwritten                                      │
│  - content (required): The complete content to write to the file                                                │
│                                                                                                                 │
│  Important: This replaces the entire file content.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [71]:
from state import DeepAgentState

In [72]:
from typing import Annotated
from langgraph.prebuilt import InjectedState
from langchain_core.tools import InjectedToolCallId, tool

@tool(description=READ_FILE_DESCRIPTION)
def ls(
    state : Annotated[DeepAgentState,InjectedState]
) -> list[str]:
    """List all files in the virtual filesystem."""
    return list(state.get("files",{}).keys())

In [73]:
@tool(description=READ_FILE_DESCRIPTION,parse_docstring=True)
def read_file(
    file_path:str,
    state:Annotated[DeepAgentState,InjectedState],
    offset:int=0,
    limit:int=2000
) -> str:
    """Read file content from virtual filesystem with optional offset and limit.

    Args:
        file_path: Path to the file to read
        state: Agent state containing virtual filesystem (injected in tool node)
        offset: Line number to start reading from (default: 0)
        limit: Maximum number of lines to read (default: 2000)

    Returns:
        Formatted file content with line numbers, or error message if file not found
    """
    files = state.get("files",{})
    if file_path not in files:
        return f"Error file : '{file_path}' not found"

    content = files[file_path]
    if not content:
        return "System reminder : File exists but has empty contents"
    
    lines = content.splitlines()
    start_idx = offset
    end_idx = min(start_idx+limit,len(lines))
    if start_idx >= len(lines):
        return f"Error: Line offset {offset} exceeds file length ({len(lines)} lines)"
    result_lines = []
    for i in range(start_idx, end_idx):
        line_content = lines[i][:2000]  # Truncate long lines
        result_lines.append(f"{i + 1:6d}\t{line_content}")

    return "\n".join(result_lines)


In [74]:
from langchain_core.messages import ToolMessage
from langgraph.types import Command

@tool(description=WRITE_FILE_DESCRIPTION)
def write_file(
    file_path:str,
    content:str,
    state : Annotated[DeepAgentState,InjectedState],
    tool_call_id : Annotated[str,InjectedToolCallId]
) -> Command:
    """Write content to a file in the virtual filesystem.

    Args:
        file_path: Path where the file should be created/updated
        content: Content to write to the file
        state: Agent state containing virtual filesystem (injected in tool node)
        tool_call_id: Tool call identifier for message response (injected in tool node)

    Returns:
        Command to update agent state with new file content
    """
    files = state.get("files",{})
    files[file_path] = content
    return Command(
        update={
            "files":files,
            "messages":[
                
                    ToolMessage(f"Updated file : {file_path}",tool_call_id=tool_call_id)
                
            ]
        }
    )


In [75]:
# Mock search result
search_result = """The Model Context Protocol (MCP) is an open standard protocol developed 
by Anthropic to enable seamless integration between AI models and external systems like 
tools, databases, and other services. It acts as a standardized communication layer, 
allowing AI models to access and utilize data from various sources in a consistent and 
efficient manner. Essentially, MCP simplifies the process of connecting AI assistants 
to external services by providing a unified language for data exchange. """

@tool(parse_docstring=True)
def web_research(
    query:str,
)->str:
    """Search the web for information on a specific topic.

    This tool performs web searches and returns relevant results
    for the given query. Use this when you need to gather information from
    the internet about any topic.

    Args:
        query: The search query string. Be specific and clear about what
               information you're looking for.

    Returns:
        Search results from search engine.

    Example:
        web_research("machine learning applications in healthcare")
    """
    return search_result

In [76]:
# File usage instructions
FILE_USAGE_INSTRUCTIONS = """You have access to a virtual file system to help you retain and save context.                                  
                                                                                                                
## Workflow Process                                                                                            
1. **Orient**: Use ls() to see existing files before starting work                                             
2. **Save**: Use write_file() to store the user's request so that we can keep it for later                     
3. **Read**: Once you are satisfied with the collected sources, read the saved file and use it to ensure that you directly answer the user's question."""

# Add mock research instructions
SIMPLE_RESEARCH_INSTRUCTIONS = """IMPORTANT: Just make a single call to the web_research tool and use the result provided by the tool to answer the user's question."""

## Full Prompt
INSTRUCTIONS = (
    FILE_USAGE_INSTRUCTIONS + "\n\n" + "=" * 80 + "\n\n" +
    SIMPLE_RESEARCH_INSTRUCTIONS
)
show_prompt(INSTRUCTIONS)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│  You have access to a virtual file system to help you retain and save context.                                  │
│                                                                                                                 │
│  ## Workflow Process                                                                                            │
│  1. **Orient**: Use ls() to see existing files before starting work                                             │
│  2. **Save**: Use write_file() to store the user's request so that we can keep it for later                     │
│  3. **Read**: Once you are satisfied with the collected sources, read the saved file and use it to ensure that  │
│  you directly answer the user's question.                                                                       │
│                                                                                                                 │
│  ================================================================================                               │
│                                                                                                                 │
│  IMPORTANT: Just make a single call to the web_research tool and use the result provided by the tool to answer  │
│  the user's question.                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [77]:
tools = [ls,read_file,write_file,web_research]

In [78]:
from langchain.agents import create_agent
agent = create_agent(
    model=llm,
    tools=tools,
    state_schema=DeepAgentState,
    system_prompt=INSTRUCTIONS
)

In [79]:
from utils import format_messages
result = agent.invoke(
    {
        "files":{},
        "messages":[
            {
                "role":"user",
                "content":"Give me an overview of Model Context Protocol (MCP)."
            }
        ]
    }
)

format_messages(result["messages"])

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Give me an overview of Model Context Protocol (MCP).                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔧 Tool Call: ls                                                                                                │
│    Args: {                                                                                                      │
│   "file_path": "."                                                                                              │
│ }                                                                                                               │
│    ID: sd8y6p5s                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔧 Tool Call: write_file                                                                                        │
│    Args: {                                                                                                      │
│   "file_path": "user_request.txt",                                                                              │
│   "content": "Give me an overview of Model Context Protocol (MCP)."                                             │
│ }                                                                                                               │
│    ID: g6ijck8w                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Updated file : user_request.txt                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔧 Tool Call: web_research                                                                                      │
│    Args: {                                                                                                      │
│   "query": "Model Context Protocol MCP overview specification"                                                  │
│ }                                                                                                               │
│    ID: ejet77kf                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ The Model Context Protocol (MCP) is an open standard protocol developed                                         │
│ by Anthropic to enable seamless integration between AI models and external systems like                         │
│ tools, databases, and other services. It acts as a standardized communication layer,                            │
│ allowing AI models to access and utilize data from various sources in a consistent and                          │
│ efficient manner. Essentially, MCP simplifies the process of connecting AI assistants                           │
│ to external services by providing a unified language for data exchange.                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔧 Tool Call: read_file                                                                                         │
│    Args: {                                                                                                      │
│   "file_path": "user_request.txt"                                                                               │
│ }                                                                                                               │
│    ID: mntx1di2                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│      1  Give me an overview of Model Context Protocol (MCP).                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ The **Model Context Protocol (MCP)** is an open-source standard developed by Anthropic designed to solve one of │
│ the biggest challenges in AI development: **connecting AI models to external data sources, tools, and           │
│ services.**                                                                                                     │
│                                                                                                                 │
│ Think of MCP as the **"USB port" for AI**. Just as USB standardized how hardware peripherals (like mice,        │
│ keyboards, and drives) connect to computers, MCP standardizes how AI models connect to data repositories,       │
│ development environments, and APIs.                                                                             │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 1. Why is MCP Needed?                                                                                       │
│ Before MCP, integrating an AI model with external systems was highly fragmented:                                │
│ * **Custom Integrations:** Developers had to write bespoke, non-reusable code to connect an AI assistant to a   │
│ specific database, GitHub repository, or local filesystem.                                                      │
│ * **Lack of Portability:** A tool built for one AI platform or IDE could not easily be used in another without  │
│ rewriting the integration.                                                                                      │
│ * **Maintenance Overhead:** As APIs changed or new models were released, maintaining these custom connections   │
│ became increasingly complex.                                                                                    │
│                                                                                                                 │
│ MCP solves this by providing a **unified, standardized protocol** for secure, bi-directional communication      │
│ between AI models and external data/tools.                                                                      │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 2. Core Architecture                                                                                        │
│ The MCP architecture is built around three main components:                                                     │
│                                                                                                                 │
│ 1. **MCP Clients:**                                                                                             │
│    These are the applications that host the AI model and orchestrate the interaction. Examples include the      │
│ **Claude Desktop App**, IDEs (like VS Code or Cursor), or custom AI applications. The client is responsible for │
│ managing permissions and routing requests.                                                                      │
│ 2. **MCP Servers:**                                                                                             │
│    These are lightweight services that expose specific capabilities (like reading a local directory, querying a │
│ Postgres database, or searching the web) via the standa